In [ ]:
import os
import json
import numpy as np
import pandas as pd
from datetime import datetime

np.random.seed(42)
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

print("=" * 70)
print("GENERATING COMPREHENSIVE INTAIN HACKATHON DATASET")
print("=" * 70)

# -------------------------------------------------------------
# 1. Generate Static Attributes (loan_static_attributes.csv)
# -------------------------------------------------------------
N_LOANS_TRAIN = 4000
N_LOANS_TEST = 1000
TOTAL_LOANS = N_LOANS_TRAIN + N_LOANS_TEST

loan_ids = [f"LOAN_{i:06d}" for i in range(1, TOTAL_LOANS + 1)]
states = ["CA", "TX", "NY", "FL", "IL", "PA", "OH", "WA", "NC", "GA"]
purposes = ["Purchase", "Refinance_Rate_Term", "Refinance_CashOut"]
occupancies = ["Owner_Occupied", "Investment", "Second_Home"]
properties = ["Single_Family", "Condo", "Multi_Family", "Townhouse"]
servicers = ["Servicer_Apex", "Servicer_Beacon", "Servicer_Crest", "Servicer_Delta"]
doc_statuses = ["Complete", "Missing_Tax_Doc", "Missing_Income", "Pending_Audit"]
systems = ["CoreBanking_V1", "CoreBanking_V2", "OriginationPortal"]

credit_bands = ["Poor", "Fair", "Good", "Excellent"]
credit_weights = [0.15, 0.35, 0.35, 0.15]
ltv_bands = ["<60%", "60-80%", "80-95%", ">95%"]
ltv_weights = [0.20, 0.45, 0.25, 0.10]
dti_bands = ["<20%", "20-35%", "36-43%", ">43%"]
dti_weights = [0.15, 0.40, 0.30, 0.15]

# Assign origination dates (2020-01 to 2022-12)
orig_dates = pd.date_range("2020-01-01", "2022-12-01", freq="MS")

static_data = {
    "loan_id": loan_ids,
    "origination_month": np.random.choice(orig_dates.strftime("%Y-%m"), size=TOTAL_LOANS),
    "original_balance": np.round(np.random.uniform(100000, 750000, size=TOTAL_LOANS), 2),
    "interest_rate": np.round(np.random.normal(5.5, 1.2, size=TOTAL_LOANS).clip(2.5, 11.0), 3),
    "credit_score_band": np.random.choice(credit_bands, p=credit_weights, size=TOTAL_LOANS),
    "ltv_band": np.random.choice(ltv_bands, p=ltv_weights, size=TOTAL_LOANS),
    "dti_band": np.random.choice(dti_bands, p=dti_weights, size=TOTAL_LOANS),
    "state": np.random.choice(states, size=TOTAL_LOANS),
    "loan_purpose": np.random.choice(purposes, size=TOTAL_LOANS),
    "occupancy_type": np.random.choice(occupancies, p=[0.75, 0.15, 0.10], size=TOTAL_LOANS),
    "property_type": np.random.choice(properties, size=TOTAL_LOANS),
    "servicer_name": np.random.choice(servicers, size=TOTAL_LOANS),
    "document_status": np.random.choice(doc_statuses, p=[0.88, 0.05, 0.05, 0.02], size=TOTAL_LOANS),
    "source_system": np.random.choice(systems, size=TOTAL_LOANS),
    "last_updated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}
df_static = pd.DataFrame(static_data)
df_static.to_csv(os.path.join(DATA_DIR, "loan_static_attributes.csv"), index=False)
print(f"[1/8] Created loan_static_attributes.csv ({len(df_static)} records)")

# -------------------------------------------------------------
# 2. Longitudinal Monthly Performance & Realistic Targets
# -------------------------------------------------------------
print("Simulating longitudinal loan monthly trajectories...")
MAX_MONTHS = 36
all_monthly_records = []

for idx, row in df_static.iterrows():
    lid = row["loan_id"]
    orig_m = datetime.strptime(row["origination_month"], "%Y-%m")
    orig_bal = row["original_balance"]
    rate = row["interest_rate"]
    credit = row["credit_score_band"]
    
    # Intrinsic risk multipliers
    risk_mult = {"Poor": 3.0, "Fair": 1.5, "Good": 0.6, "Excellent": 0.2}[credit]
    curr_bal = orig_bal
    dpd = 0
    state = "Current"
    
    loan_trajectory = []
    
    for m in range(MAX_MONTHS):
        rep_date = orig_m + pd.DateOffset(months=m)
        rep_month_str = rep_date.strftime("%Y-%m")
        rem_term = max(360 - m, 0)
        
        # Balance amortization
        monthly_principal = (orig_bal / 360) * (1 + 0.02 * m)
        curr_bal = max(curr_bal - monthly_principal, 0.0)
        
        # State transitions
        mod_flag = 0
        prepay_flag = 0
        default_flag = 0
        
        # Anomaly injection: Intentional balance spike / date mismatch
        if np.random.rand() < 0.008:
            curr_bal = orig_bal * 1.15  # Balance mismatch exception
            
        # Markov state updates based on risk profile
        rand_val = np.random.rand()
        if state == "Current":
            if rand_val < (0.005 * risk_mult):
                state = "30_DPD"; dpd = 30
            elif rand_val < (0.005 * risk_mult + 0.015):
                state = "Prepaid"; prepay_flag = 1; curr_bal = 0.0
            else:
                dpd = 0
        elif state == "30_DPD":
            if rand_val < 0.40:
                state = "Current"; dpd = 0
            elif rand_val < 0.70:
                state = "60_DPD"; dpd = 60
            elif rand_val < 0.75:
                mod_flag = 1; state = "Current"; dpd = 0
        elif state == "60_DPD":
            if rand_val < 0.25:
                state = "30_DPD"; dpd = 30
            elif rand_val < 0.70:
                state = "90_DPD"; dpd = 90
            elif rand_val < 0.75:
                mod_flag = 1; state = "Current"; dpd = 0
        elif state == "90_DPD":
            if rand_val < 0.15:
                state = "60_DPD"; dpd = 60
            elif rand_val < 0.65:
                state = "Defaulted"; default_flag = 1; dpd = 120
            elif rand_val < 0.70:
                mod_flag = 1; state = "Current"; dpd = 0
                
        loss_sev = "None"
        if default_flag == 1:
            loss_sev = np.random.choice(["Low_<20%", "Medium_20-40%", "High_>40%"])
            
        loan_trajectory.append({
            "loan_id": lid,
            "month_index": m + 1,
            "reporting_month": rep_month_str,
            "origination_month": row["origination_month"],
            "loan_age_months": m,
            "remaining_term_months": rem_term,
            "original_balance": orig_bal,
            "current_balance": round(curr_bal, 2),
            "interest_rate": rate,
            "credit_score_band": credit,
            "ltv_band": row["ltv_band"],
            "dti_band": row["dti_band"],
            "state": row["state"],
            "loan_purpose": row["loan_purpose"],
            "occupancy_type": row["occupancy_type"],
            "property_type": row["property_type"],
            "servicer_name": row["servicer_name"],
            "current_status": state,
            "days_past_due": dpd,
            "modification_flag": mod_flag,
            "prepayment_flag": prepay_flag,
            "default_flag": default_flag,
            "loss_severity_band": loss_sev,
            "document_status": row["document_status"],
            "source_system": row["source_system"],
            "last_updated_at": rep_date.strftime("%Y-%m-%d %H:%M:%S")
        })
        
        if state in ["Prepaid", "Defaulted"]:
            break

    # Compute forward-looking target variables over trajectory
    traj_len = len(loan_trajectory)
    for i in range(traj_len):
        # 1. Next State
        loan_trajectory[i]["next_state"] = loan_trajectory[i+1]["current_status"] if i+1 < traj_len else "Terminated"
        
        # 2. Next 3M Delinquency (DPD >= 30)
        window_3m = loan_trajectory[i+1:min(i+4, traj_len)]
        loan_trajectory[i]["next_3m_delinquency_flag"] = int(any(r["days_past_due"] >= 30 for r in window_3m)) if window_3m else 0
        
        # 3. Next 6M Delinquency (DPD >= 30)
        window_6m = loan_trajectory[i+1:min(i+7, traj_len)]
        loan_trajectory[i]["next_6m_delinquency_flag"] = int(any(r["days_past_due"] >= 30 for r in window_6m)) if window_6m else 0
        
        # 4. Next 12M Default
        window_12m = loan_trajectory[i+1:min(i+13, traj_len)]
        loan_trajectory[i]["next_12m_default_flag"] = int(any(r["default_flag"] == 1 for r in window_12m)) if window_12m else 0
        
        # 5. Next 12M Prepayment
        loan_trajectory[i]["next_12m_prepayment_flag"] = int(any(r["prepayment_flag"] == 1 for r in window_12m)) if window_12m else 0
        
        # 6. Exception required & Exception type
        rec = loan_trajectory[i]
        ex_req = 0
        ex_type = "None"
        if rec["current_balance"] > rec["original_balance"] and rec["modification_flag"] == 0:
            ex_req = 1; ex_type = "Data_Mismatch"
        elif rec["document_status"] in ["Missing_Tax_Doc", "Missing_Income"]:
            ex_req = 1; ex_type = "Document_Gap"
        elif rec["days_past_due"] >= 90 and rec["current_status"] == "Current":
            ex_req = 1; ex_type = "Policy_Violation"
            
        loan_trajectory[i]["exception_required"] = ex_req
        loan_trajectory[i]["exception_type"] = ex_type

    all_monthly_records.extend(loan_trajectory)

df_all = pd.DataFrame(all_monthly_records)

# Partition into Train and Test based on Loan IDs
train_lids = set(loan_ids[:N_LOANS_TRAIN])
df_train = df_all[df_all["loan_id"].isin(train_lids)].copy()
df_test = df_all[~df_all["loan_id"].isin(train_lids)].copy()

# Drop future targets from test dataset to match official competition schema
test_target_cols = [
    "next_3m_delinquency_flag", "next_6m_delinquency_flag", 
    "next_12m_default_flag", "next_12m_prepayment_flag", 
    "next_state", "exception_required", "exception_type"
]
df_test_unlabeled = df_test.drop(columns=test_target_cols)

df_train.to_csv(os.path.join(DATA_DIR, "loan_monthly_performance_train.csv"), index=False)
df_test_unlabeled.to_csv(os.path.join(DATA_DIR, "loan_monthly_performance_test.csv"), index=False)
print(f"[2/8] Created loan_monthly_performance_train.csv ({len(df_train)} rows)")
print(f"[3/8] Created loan_monthly_performance_test.csv ({len(df_test_unlabeled)} rows)")

# -------------------------------------------------------------
# 3. Servicer Updates (servicer_updates.csv)
# -------------------------------------------------------------
servicer_reconcile = df_train.sample(n=1500, random_state=42)[
    ["loan_id", "reporting_month", "current_balance", "days_past_due", "current_status"]
].copy()
# Inject intentional discrepancy for conflict detection
servicer_reconcile["current_balance"] = servicer_reconcile["current_balance"] * np.random.choice([1.0, 1.05, 0.95], p=[0.85, 0.08, 0.07], size=len(servicer_reconcile))
servicer_reconcile["servicer_reported_date"] = datetime.now().strftime("%Y-%m-%d")
servicer_reconcile.to_csv(os.path.join(DATA_DIR, "servicer_updates.csv"), index=False)
print(f"[4/8] Created servicer_updates.csv ({len(servicer_reconcile)} records)")

# -------------------------------------------------------------
# 4. Macro Scenarios (macro_scenarios.csv)
# -------------------------------------------------------------
macro_df = pd.DataFrame({
    "scenario_name": ["Base", "Adverse_Credit", "High_Prepayment"],
    "rate_shock_bps": [0, 200, -200],
    "unemployment_shock_pct": [0.0, 3.5, -0.5],
    "hpi_growth_pct": [3.0, -10.0, 6.0],
    "default_multiplier": [1.0, 2.5, 0.8],
    "prepayment_multiplier": [1.0, 0.4, 3.0]
})
macro_df.to_csv(os.path.join(DATA_DIR, "macro_scenarios.csv"), index=False)
print(f"[5/8] Created macro_scenarios.csv (3 scenarios)")

# -------------------------------------------------------------
# 5. Data Dictionary (data_dictionary.md)
# -------------------------------------------------------------
data_dict_content = """# Loan Performance Engine Data Dictionary

## Static & Panel Fields
- **loan_id**: Unique alpha-numeric identifier for each mortgage loan.
- **month_index**: Month sequence number since origination.
- **reporting_month**: Calendar month of performance record (YYYY-MM).
- **origination_month**: Origination date (YYYY-MM).
- **loan_age_months**: Months elapsed since origination.
- **original_balance**: Initial principal disbursed at closing.
- **current_balance**: Outstanding scheduled unpaid principal balance (UPB).
- **interest_rate**: Note interest rate in percentage.
- **credit_score_band**: Borrower FICO score categorized into Poor (<620), Fair (620-679), Good (680-739), Excellent (740+).
- **ltv_band**: Loan-to-Value ratio category at origination.
- **dti_band**: Debt-to-Income ratio category at origination.
- **current_status**: Monthly payment status (Current, 30_DPD, 60_DPD, 90_DPD, Prepaid, Defaulted).
- **days_past_due**: Days past due on monthly mortgage installment.
- **modification_flag**: Binary indicator (1 if modified, 0 otherwise).
- **prepayment_flag**: Binary indicator (1 if loan prepaid in full this month).
- **default_flag**: Binary indicator (1 if loan liquidated or charged-off this month).

## Target Definitions
- **next_3m_delinquency_flag**: 1 if loan reaches >=30 DPD in the subsequent 3 months.
- **next_6m_delinquency_flag**: 1 if loan reaches >=30 DPD in the subsequent 6 months.
- **next_12m_default_flag**: 1 if loan experiences default/charge-off in next 12 months.
- **next_12m_prepayment_flag**: 1 if loan prepays in full in next 12 months.
- **next_state**: Expected status in the subsequent month.
- **exception_required**: 1 if loan record fails data integrity or compliance audits.
- **exception_type**: Categorical reason (Data_Mismatch, Document_Gap, Policy_Violation, None).
"""
with open(os.path.join(DATA_DIR, "data_dictionary.md"), "w") as f:
    f.write(data_dict_content)
print(f"[6/8] Created data_dictionary.md")

# -------------------------------------------------------------
# 6. Validation Rules (validation_rules.json)
# -------------------------------------------------------------
validation_rules = {
    "balance_consistency": {
        "rule": "current_balance <= original_balance unless modification_flag == 1",
        "severity": "HIGH",
        "exception_type": "Data_Mismatch"
    },
    "date_chronology": {
        "rule": "reporting_month >= origination_month",
        "severity": "CRITICAL",
        "exception_type": "Data_Mismatch"
    },
    "dpd_status_consistency": {
        "rule": "days_past_due == 0 when current_status == 'Current'",
        "severity": "MEDIUM",
        "exception_type": "Policy_Violation"
    },
    "document_completeness": {
        "rule": "document_status == 'Complete'",
        "severity": "LOW",
        "exception_type": "Document_Gap"
    },
    "terminal_balance": {
        "rule": "current_balance == 0 when prepayment_flag == 1 or default_flag == 1",
        "severity": "HIGH",
        "exception_type": "Data_Mismatch"
    }
}
with open(os.path.join(DATA_DIR, "validation_rules.json"), "w") as f:
    json.dump(validation_rules, f, indent=4)
print(f"[7/8] Created validation_rules.json")

# -------------------------------------------------------------
# 7. Submission Template (submission_template.csv)
# -------------------------------------------------------------
sub_template = pd.DataFrame({
    "loan_id": df_test_unlabeled["loan_id"].head(5),
    "reporting_month": df_test_unlabeled["reporting_month"].head(5),
    "probability_default": [0.0] * 5,
    "probability_delinquency_3m": [0.0] * 5,
    "probability_prepayment": [0.0] * 5,
    "predicted_next_state": ["Current"] * 5,
    "exception_type": ["None"] * 5,
    "anomaly_score": [0.0] * 5,
    "top_drivers": ["None"] * 5,
    "action": ["PASS"] * 5,
    "confidence": [0.95] * 5
})
sub_template.to_csv(os.path.join(DATA_DIR, "submission_template.csv"), index=False)
print(f"[8/8] Created submission_template.csv")

print("=" * 70)
print("ALL 8 ASSETS GENERATED & VERIFIED IN 'data/' DIRECTORY")
print("=" * 70)

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, brier_score_loss, f1_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = "data"

print("\n" + "=" * 60)
print("SECTION 0: DYNAMIC DATA LOADING")
print("=" * 60)

# Flexibly load data if it exists, matching official Intain filenames
train_path = os.path.join(DATA_DIR, "loan_monthly_performance_train.csv")
test_path = os.path.join(DATA_DIR, "loan_monthly_performance_test.csv")

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)
print(f"Loaded Training Data: {df_train.shape}")
print(f"Loaded Test Data: {df_test.shape}")

print("\n" + "=" * 60)
print("TASK 1: DATA INTELLIGENCE & PROFILING")
print("=" * 60)

# 1. Missingness & Quality Scoring
missing_pct = df_train.isnull().mean() * 100
print("Top Missing Columns (%):\n", missing_pct[missing_pct > 0].sort_values(ascending=False).head())

# Record-level Data Quality Score (0 to 1)
df_train['dq_score'] = 1.0 - (df_train.isnull().sum(axis=1) / df_train.shape[1])

# 2. Relationship Breaks (e.g., Balance > Original)
balance_breaks = df_train[df_train['current_balance'] > df_train['original_balance']]
print(f"Detected {len(balance_breaks)} records where current balance exceeds original balance.")

print("\n" + "=" * 60)
print("TASK 2: FEATURE ENGINEERING & TIME-AWARE MODELING")
print("=" * 60)

# 1. Feature Engineering (Dynamically applied to both Train and Test)
def engineer_features(df):
    df = df.copy()
    df['balance_ratio'] = df['current_balance'] / df['original_balance'].replace(0, 1)
    df['is_modified'] = df['modification_flag'].fillna(0)
    
    # Encode categoricals dynamically
    cat_cols = df.select_dtypes(include=['object', 'category']).columns
    # Exclude targets and IDs from encoding
    exclude = ['loan_id', 'reporting_month', 'origination_month', 'next_state', 'exception_type']
    encode_cols = [c for c in cat_cols if c not in exclude]
    
    for col in encode_cols:
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))
    return df

df_train_fe = engineer_features(df_train)

# 2. Time-Aware Split (No random row-level splitting)
# Calculate the 80th percentile of unique reporting months for a strict chronological cutoff
unique_months = sorted(df_train_fe['reporting_month'].unique())
cutoff_idx = int(len(unique_months) * 0.8)
cutoff_month = unique_months[cutoff_idx]

train_split = df_train_fe[df_train_fe['reporting_month'] <= cutoff_month]
val_split = df_train_fe[df_train_fe['reporting_month'] > cutoff_month]
print(f"Time-Aware Split Cutoff: {cutoff_month}")
print(f"Training set: {train_split.shape[0]} rows | Validation set: {val_split.shape[0]} rows")

# 3. Define Features and Targets
target_cols = [
    'next_3m_delinquency_flag', 'next_6m_delinquency_flag', 
    'next_12m_default_flag', 'next_12m_prepayment_flag', 
    'next_state', 'exception_required', 'exception_type'
]
features = [c for c in train_split.columns if c not in target_cols + ['loan_id', 'reporting_month', 'origination_month']]

X_train = train_split[features].fillna(0)
X_val = val_split[features].fillna(0)

# 4. Train Supervised Non-LLM Models (Default & Prepayment)
models = {}
for target in ['next_12m_default_flag', 'next_12m_prepayment_flag']:
    print(f"\nTraining Model: {target}")
    y_train = train_split[target].fillna(0)
    y_val = val_split[target].fillna(0)
    
    # Handle Class Imbalance via sample weights
    class_counts = np.bincount(y_train.astype(int))
    weight_val = class_counts[0] / max(class_counts[1], 1)
    sample_weights = np.where(y_train == 1, weight_val, 1.0)
    
    # Base Model (HistGradientBoosting handles mixed data well)
    base_model = HistGradientBoostingClassifier(random_state=42, max_iter=100)
    base_model.fit(X_train, y_train, sample_weight=sample_weights)
    
    # Model Calibration
    calibrated_model = CalibratedClassifierCV(estimator=base_model, method='isotonic', cv='prefit')
    calibrated_model.fit(X_val, y_val)
    models[target] = calibrated_model
    
    # Evaluation
    preds = calibrated_model.predict(X_val)
    probs = calibrated_model.predict_proba(X_val)[:, 1]
    
    print(f"ROC-AUC: {roc_auc_score(y_val, probs):.4f}")
    print(f"Brier Score: {brier_score_loss(y_val, probs):.4f}")
    print(f"F1 Score: {f1_score(y_val, preds):.4f}")

# 5. Train Multi-Class Next State Model
print("\nTraining Model: next_state (Multi-class Transition)")
y_train_state = LabelEncoder().fit_transform(train_split['next_state'].astype(str))
y_val_state = LabelEncoder().fit_transform(val_split['next_state'].astype(str))

state_model = HistGradientBoostingClassifier(random_state=42, max_iter=100)
state_model.fit(X_train, y_train_state)
models['next_state'] = state_model
print("Multi-class state model trained successfully.")

In [ ]:
!pip install lifelines

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter, CoxPHFitter
from sklearn.ensemble import IsolationForest

print("\n" + "=" * 60)
print("TASK 3: SURVIVAL & TIME-TO-EVENT MODELING")
print("=" * 60)

# 1. Prepare Survival Data (One row per loan representing its terminal state)
survival_df = df_train_fe.groupby('loan_id').agg(
    duration=('loan_age_months', 'max'),
    event_observed=('default_flag', 'max'),
    credit_score_band=('credit_score_band', 'last') 
).reset_index()

# 2. Kaplan-Meier Baseline Survival Curve
kmf = KaplanMeierFitter()
kmf.fit(survival_df['duration'], event_observed=survival_df['event_observed'])

plt.figure(figsize=(10, 5))
kmf.plot_survival_function()
plt.title("Kaplan-Meier Survival Curve: Time to Default")
plt.xlabel("Months Since Origination")
plt.ylabel("Survival Probability")
plt.tight_layout()
plt.savefig("task3_kaplan_meier_baseline.png")
print("Saved Kaplan-Meier baseline plot to 'task3_kaplan_meier_baseline.png'.")
plt.close()

# 3. Cox Proportional Hazards Model
cox_df = survival_df[['duration', 'event_observed', 'credit_score_band']].dropna()
cph = CoxPHFitter()
cph.fit(cox_df, duration_col='duration', event_col='event_observed')
print("\nCox Proportional Hazards Summary:")
cph.print_summary(decimals=3)

print("\n" + "=" * 60)
print("TASK 4: ANOMALY & EXCEPTION DETECTION")
print("=" * 60)

# 1. Unsupervised ML Anomaly Detection
anomaly_features = [col for col in features if col not in ['exception_required', 'exception_type', 'dq_score']]
X_anomaly = X_val[anomaly_features].fillna(0)

iso_forest = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
iso_forest.fit(X_train[anomaly_features].fillna(0))

# Normalize scores (0 to 1, where 1 is highly anomalous)
raw_scores = iso_forest.decision_function(X_anomaly)
normalized_scores = (raw_scores.max() - raw_scores) / (raw_scores.max() - raw_scores.min())

# 2. Hybrid Rule + ML Scoring
anomaly_df = val_split[['loan_id', 'reporting_month', 'exception_required', 'exception_type']].copy()
anomaly_df['ml_anomaly_score'] = normalized_scores
anomaly_df['hybrid_risk_score'] = (anomaly_df['ml_anomaly_score'] * 0.4) + (anomaly_df['exception_required'] * 0.6)

# 3. Compile Reviewer-Ready Examples
reviewer_examples = anomaly_df[
    (anomaly_df['exception_required'] == 1) | (anomaly_df['hybrid_risk_score'] > 0.7)
].sort_values(by='hybrid_risk_score', ascending=False).head(25)

reviewer_examples['anomaly_drivers'] = np.where(
    reviewer_examples['exception_type'] == 'Data_Mismatch', "Current Balance > Original Balance",
    np.where(reviewer_examples['exception_type'] == 'Document_Gap', "Missing Required Documentation",
    np.where(reviewer_examples['exception_type'] == 'Policy_Violation', "90+ DPD while flagged as Current",
    "High ML Isolation Forest Score"))
)

reviewer_examples.to_csv("reviewer_ready_anomalies.csv", index=False)
print(f"\nGenerated {len(reviewer_examples)} reviewer-ready anomaly examples.")
print("Saved anomalies to 'reviewer_ready_anomalies.csv'.")

print("\n--- Top 3 Anomalies ---")
print(reviewer_examples[['loan_id', 'exception_type', 'hybrid_risk_score', 'anomaly_drivers']].head(3).to_string(index=False))

In [ ]:
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

print("\n" + "=" * 60)
print("TASK 5: SCENARIO & STRESS SIMULATION")
print("=" * 60)

# 1. Load Macro Scenarios
scenario_df = pd.read_csv(os.path.join(DATA_DIR, "macro_scenarios.csv"))
print("Loaded Macro Scenarios:")
print(scenario_df[['scenario_name', 'default_multiplier', 'prepayment_multiplier']].to_string(index=False))

# 2. Base Portfolio Predictions
base_default_probs = models['next_12m_default_flag'].predict_proba(X_val)[:, 1]
base_prepay_probs = models['next_12m_prepayment_flag'].predict_proba(X_val)[:, 1]

# 3. Apply Stress Multipliers & Segment by State
# We use the raw 'state' column from val_split for the business segment view
results = []
for _, row in scenario_df.iterrows():
    scenario = row['scenario_name']
    d_mult = row['default_multiplier']
    p_mult = row['prepayment_multiplier']
    
    # Project probabilities (capped at 1.0)
    proj_default = np.clip(base_default_probs * d_mult, 0, 1)
    proj_prepay = np.clip(base_prepay_probs * p_mult, 0, 1)
    
    # Create segment view
    segment_view = val_split[['state']].copy()
    segment_view['proj_default_prob'] = proj_default
    segment_view['proj_prepay_prob'] = proj_prepay
    segment_view['scenario'] = scenario
    
    results.append(segment_view)

# Aggregate and save report
scenario_results = pd.concat(results)
scenario_summary = scenario_results.groupby(['scenario', 'state']).mean().reset_index()
scenario_summary.to_csv("scenario_report.csv", index=False)

print("\nGenerated Scenario Stress Report (Top 5 rows):")
print(scenario_summary.head().to_string(index=False))


print("\n" + "=" * 60)
print("TASK 6: EXPLAINABILITY LAYER & RESPONSIBLE AI")
print("=" * 60)

# 1. Train Surrogate Explainer Model (Random Forest for SHAP compatibility)
print("Training explainability surrogate model...")
explainer_model = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
y_val_default = val_split['next_12m_default_flag'].fillna(0)
explainer_model.fit(X_val, y_val_default)

# 2. Global Feature Importance
print("Calculating SHAP values...")
explainer = shap.TreeExplainer(explainer_model)
shap_obj = explainer(X_val)

# Handle potential 3D array return from RF
if len(shap_obj.shape) == 3:
    shap_values_pos = shap_obj[:, :, 1]
else:
    shap_values_pos = shap_obj

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_pos.values, X_val, show=False)
plt.title("Global Explanations: Top Drivers of Default")
plt.tight_layout()
plt.savefig("task6_global_shap_summary.png", dpi=300)
print("Global feature importance saved to 'task6_global_shap_summary.png'.")
plt.close()

# 3. Local Explanation (Waterfall)
loan_idx = 0 
plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values_pos[loan_idx], show=False)
plt.title(f"Local Explanation: Drivers of Default (Validation Index {loan_idx})")
plt.tight_layout()
plt.savefig("task6_local_shap_waterfall.png", dpi=300)
print("Local explanation waterfall plot saved to 'task6_local_shap_waterfall.png'.")
plt.close()

# 4. Error Analysis (False Positives / False Negatives)
preds = explainer_model.predict(X_val)
probs = explainer_model.predict_proba(X_val)[:, 1]

uncertain_mask = (probs > 0.40) & (probs < 0.60)
tn, fp, fn, tp = confusion_matrix(y_val_default, preds).ravel()

print("\n--- Error Analysis & Model Uncertainty ---")
print(f"Low Confidence / Uncertain Predictions (0.4-0.6 Prob): {uncertain_mask.sum()} out of {len(probs)} loans")
print(f"False Positives (Predicted Default, Actually Paid): {fp}")
print(f"False Negatives (Predicted Paid, Actually Defaulted): {fn}")

In [ ]:
import os
import google.generativeai as genai
from kaggle_secrets import UserSecretsClient
import pandas as pd
import numpy as np
from datetime import datetime

print("\n" + "=" * 60)
print("TASK 7: ADVANCED RAG & LLM COPILOT (ZERO HALLUCINATION)")
print("=" * 60)

# 1. Dynamic RAG Data Dictionary Retrieval (No Hardcoding)
dict_filename = os.path.join(DATA_DIR, "data_dictionary.md")
data_dictionary = {}

if os.path.exists(dict_filename):
    print(f"Loading official knowledge base from {dict_filename}...")
    with open(dict_filename, 'r') as file:
        for line in file:
            if ':' in line:
                key, val = line.split(':', 1)
                data_dictionary[key.strip().replace("**", "").replace("- ", "")] = val.strip()
else:
    print("Notice: data_dictionary.md not found. Ensure the data generator ran successfully.")

# 2. Authenticate with Kaggle Secrets
try:
    user_secrets = UserSecretsClient()
    api_key = user_secrets.get_secret("GEMINI_API_KEY")
    genai.configure(api_key=api_key)
    model_name = "gemini-3.6-flash" 
    llm = genai.GenerativeModel(model_name)
    api_ready = True
except Exception as e:
    print("Notice: GEMINI_API_KEY not found in Kaggle Secrets. Using grounded fallback.")
    api_ready = False
    model_name = "fallback-offline"

# 3. Fetch a real anomaly from Task 4
sample_anomaly = reviewer_examples.iloc[0]
loan_id = sample_anomaly['loan_id']
anomaly_drivers = sample_anomaly['anomaly_drivers']
exception_type = sample_anomaly['exception_type']

# 4. Lightweight RAG Retrieval
retrieved_context = data_dictionary.get(exception_type, "Exception definition not found in dictionary.")

# 5. Construct Grounded Prompt
prompt = f"""
You are an AI assistant for a loan reviewer. Base your response strictly on the data provided below.
Do not invent information.

[SYSTEM CONTEXT - DATA DICTIONARY]
{exception_type}: {retrieved_context}

[LOAN DATA]
Loan ID: {loan_id}
Flags: {exception_type}
Anomaly Drivers: {anomaly_drivers}

Task: Write a concise, 2-sentence reviewer note summarizing why this loan was flagged based on the dictionary definition, and state what the human reviewer should verify.
"""

# 6. Execute API Call
if api_ready:
    try:
        generation_config = genai.types.GenerationConfig(temperature=0.0)
        response = llm.generate_content(prompt, generation_config=generation_config)
        llm_output = response.text.strip()
    except Exception as e:
        llm_output = f"API Error: {str(e)}"
else:
    llm_output = f"Loan {loan_id} triggered a {exception_type} due to {anomaly_drivers.lower()}. The reviewer must verify this against system records as per the data dictionary definition: {retrieved_context}"
    
governance_disclaimer = "\n\n[SYSTEM NOTE: This is an AI-generated recommendation. Human decision required.]"
final_output = llm_output + governance_disclaimer

# 7. Log Interaction
log_record = {
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "model": model_name,
    "temperature": 0.0,
    "prompt": prompt,
    "output": final_output
}
pd.DataFrame([log_record]).to_csv("llm_prompt_logs_rag.csv", index=False)
print("\n--- LLM Copilot Output ---")
print(final_output)

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

print("\n" + "=" * 60)
print("TASK 8: FINAL PREDICTIONS & SUBMISSION (CORRECTED)")
print("=" * 60)

# 1. Prepare Test Data & Ensure Schema Alignment
# Compute the missing dq_score on the test set identical to Task 1
df_test['dq_score'] = 1.0 - (df_test.isnull().sum(axis=1) / df_test.shape[1])

# Apply the shared feature engineering pipeline
df_test_fe = engineer_features(df_test)
X_test = df_test_fe[features].fillna(0)

# 2. Run Inference
print("Executing production inference on test dataset...")
prob_default = models['next_12m_default_flag'].predict_proba(X_test)[:, 1]
prob_prepayment = models['next_12m_prepayment_flag'].predict_proba(X_test)[:, 1]
pred_next_state = models['next_state'].predict(X_test)

# Map encoded states back to strings
state_encoder = LabelEncoder().fit(df_train_fe['next_state'].astype(str))
pred_next_state_labels = state_encoder.inverse_transform(pred_next_state)

# Generate Anomaly Scores for Test Set
test_anomaly_scores = (iso_forest.decision_function(X_test[anomaly_features]) * -1) 
test_anomaly_scores = (test_anomaly_scores - test_anomaly_scores.min()) / (test_anomaly_scores.max() - test_anomaly_scores.min())

# 3. Compile Strict Submission Format
submission = pd.DataFrame({
    'loan_id': df_test['loan_id'],
    'probability_default': prob_default,
    'probability_delinquency_3m': np.clip(prob_default * 1.5, 0, 1), # Proxy estimation
    'probability_prepayment': prob_prepayment,
    'predicted_next_state': pred_next_state_labels,
    'exception_type': np.where(test_anomaly_scores > 0.7, 'Statistical_Outlier', 'None'),
    'anomaly_score': test_anomaly_scores,
    'top_drivers': np.where(test_anomaly_scores > 0.7, 'High ML Isolation Forest Score', 'None'),
    'action': np.where(test_anomaly_scores > 0.7, 'REVIEW', 'PASS'),
    'confidence': np.clip(1.0 - test_anomaly_scores, 0.5, 0.99)
})

submission.to_csv("submission.csv", index=False)
print(f"Successfully generated 'submission.csv' ({len(submission)} rows).")
print(submission.head().to_string(index=False))

In [ ]:
import os
from scipy.stats import ks_2samp

print("\n" + "=" * 60)
print("PATCH: ADVANCED PROFILING & REPORT EXPORT")
print("=" * 60)

os.makedirs("reports", exist_ok=True)

# 1. Train/Test Drift Analysis (KS Test on balances)
ks_stat, p_value = ks_2samp(df_train['current_balance'].dropna(), df_test['current_balance'].dropna())
drift_status = "Drift Detected" if p_value < 0.05 else "Stable (No significant drift)"

# 2. Correlation Matrix extraction
numeric_cols = df_train.select_dtypes(include=[np.number]).columns
corr_matrix = df_train[numeric_cols].corr().abs()
# Extract top correlations excluding self-correlation
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
top_correlations = upper_tri.unstack().dropna().sort_values(ascending=False).head(5)

# 3. Export Data Intelligence Report
missing_pct = df_train.isnull().mean() * 100
dq_score_mean = df_train['dq_score'].mean()

di_report = f"""# Data Intelligence Report

## 1. Data Quality Overview
* **Average Record-Level Quality Score:** {dq_score_mean:.4f}
* **Records with Balance > Original:** {len(balance_breaks)}

## 2. Missingness Analysis
Top missing fields requiring imputation:
{missing_pct[missing_pct > 0].sort_values(ascending=False).head().to_string()}

## 3. Train vs. Test Distribution Drift
Evaluated using Kolmogorov-Smirnov (KS) statistic on `current_balance`.
* **KS Statistic:** {ks_stat:.4f}
* **P-Value:** {p_value:.4f}
* **Status:** {drift_status}

## 4. Top Feature Correlations
{top_correlations.to_string()}
"""

with open("reports/data_intelligence_report.md", "w") as f:
    f.write(di_report)

print("Advanced profiling complete. Exported 'reports/data_intelligence_report.md'.")

In [ ]:
print("\n" + "=" * 60)
print("PATCH: EXPLAINABILITY REPORT EXPORT")
print("=" * 60)

exp_report = f"""# Explainability & Responsible AI Report

## 1. Global & Local Drivers
* **Global Importance:** Refer to `task6_global_shap_summary.png`.
* **Local Explanation Example:** Refer to `task6_local_shap_waterfall.png`.

## 2. Model Confidence & Uncertainty
* **Total Validation Records:** {len(probs)}
* **Low Confidence Predictions (0.40 - 0.60 Probability):** {uncertain_mask.sum()}

## 3. Error Analysis (Default Model)
* **False Positives (Predicted Default, Actually Paid):** {fp}
* **False Negatives (Predicted Paid, Actually Defaulted):** {fn}
"""

with open("reports/explainability_report.md", "w") as f:
    f.write(exp_report)

print("Exported 'reports/explainability_report.md'.")

In [ ]:
print("\n" + "=" * 60)
print("PATCH: DEDICATED 3M DELINQUENCY MODEL & FINAL SUBMISSION")
print("=" * 60)

# 1. Train the dedicated 3-month delinquency model
print("Training Model: next_3m_delinquency_flag...")
y_train_3m = train_split['next_3m_delinquency_flag'].fillna(0)
y_val_3m = val_split['next_3m_delinquency_flag'].fillna(0)

# Handle class imbalance
class_counts_3m = np.bincount(y_train_3m.astype(int))
weight_val_3m = class_counts_3m[0] / max(class_counts_3m[1], 1)
sample_weights_3m = np.where(y_train_3m == 1, weight_val_3m, 1.0)

base_model_3m = HistGradientBoostingClassifier(random_state=42, max_iter=100)
base_model_3m.fit(X_train, y_train_3m, sample_weight=sample_weights_3m)

calibrated_model_3m = CalibratedClassifierCV(estimator=base_model_3m, method='isotonic', cv='prefit')
calibrated_model_3m.fit(X_val, y_val_3m)

# 2. Run Inference on Test Set
print("Executing production inference for all targets...")
prob_default = models['next_12m_default_flag'].predict_proba(X_test)[:, 1]
prob_prepayment = models['next_12m_prepayment_flag'].predict_proba(X_test)[:, 1]
prob_delinq_3m = calibrated_model_3m.predict_proba(X_test)[:, 1] # Real predictions, no longer a proxy
pred_next_state = models['next_state'].predict(X_test)

pred_next_state_labels = state_encoder.inverse_transform(pred_next_state)

# 3. Compile Strict Submission Format
submission = pd.DataFrame({
    'loan_id': df_test['loan_id'],
    'probability_default': prob_default,
    'probability_delinquency_3m': prob_delinq_3m, 
    'probability_prepayment': prob_prepayment,
    'predicted_next_state': pred_next_state_labels,
    'exception_type': np.where(test_anomaly_scores > 0.7, 'Statistical_Outlier', 'None'),
    'anomaly_score': test_anomaly_scores,
    'top_drivers': np.where(test_anomaly_scores > 0.7, 'High ML Isolation Forest Score', 'None'),
    'action': np.where(test_anomaly_scores > 0.7, 'REVIEW', 'PASS'),
    'confidence': np.clip(1.0 - test_anomaly_scores, 0.5, 0.99)
})

submission.to_csv("submission.csv", index=False)
print(f"Successfully generated final un-proxied 'submission.csv' ({len(submission)} rows).")